# 🧩 Integración: Gradio y MediaPipe

**Materiales desarrollados por Matías Barreto, 2026**  
**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**  
* **Nomenclatura Oficial:** Procesamiento Digital de Imágenes  
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital  

---

## Objetivo

En este laboratorio vamos a conectar dos herramientas que ya conocemos: **MediaPipe** para detectar puntos clave en imágenes y **Gradio** para construir interfaces web interactivas. El resultado va a ser una aplicación que cualquier persona puede usar desde el navegador, sin instalar nada.

## Al terminar este material vamos a poder:

1. Explicar qué es una **Skill** y por qué empaquetar conocimiento de forma estructurada tiene sentido.
2. Construir una interfaz web mínima con `gr.Interface` a partir de una función Python.
3. Crear layouts con control total usando `gr.Blocks`, botones y eventos.
4. Integrar un detector de MediaPipe dentro de una interfaz Gradio lista para compartir.

## Microglosario

Antes de escribir código, definimos los términos clave.

| Término | Definición | Analogía |
|---|---|---|
| **Skill** | Paquete de conocimiento estructurado y reutilizable que un agente puede cargar automáticamente | Como un libro de recetas profesional: no explicás cada vez cómo hacer una bechamel; la receta está ahí, lista para quien la necesite |
| **`gr.Interface`** | Forma rápida de envolver una función Python en una interfaz web con entradas y salidas definidas | Como enmarcar un cuadro: le das la obra (la función) y el marco se encarga de que se vea bien |
| **`gr.Blocks`** | Constructor de interfaces con control total sobre el layout, botones y eventos | Como armar un tablero de control a medida: vos elegís dónde va cada componente y qué hace |
| **Componente** | Elemento de interfaz (imagen, texto, slider) que conecta al usuario con la función Python | Como los enchufes de un equipo de audio: cada uno tiene un tipo de señal específico |
| **Evento** | Acción del usuario que dispara la ejecución de una función (click, cambio de valor, envío) | Como el timbre de la puerta: cuando suena, alguien va a atender |

## ✦ Concepto: ¿Qué es una Skill?

Antes de arrancar con el código, vamos a entender qué problema resuelve el concepto de **Skill** y por qué Hugging Face lo adoptó como estándar en su ecosistema de agentes.

### El problema: el agente no tiene contexto

Imaginemos que le pedimos a un agente de IA que publique un modelo en Hugging Face. Sin instrucciones específicas, el agente va a:

- No saber cómo autenticarse con nuestro token
- No conocer el formato de datos que usamos
- Publicar el modelo sin README, sin licencia, sin ejemplos de uso

Esos errores no son de inteligencia: son de **falta de contexto de dominio**.

### Primera respuesta: el prompt largo

Una reacción natural sería escribir un prompt detallado con todas las instrucciones:

```
Sos un experto en Hugging Face. Cuando publiques un modelo:
1. Autenticáte con HF_TOKEN
2. Validá que el dataset esté en formato CSV con columnas: text, label, split
3. Entrená con learning_rate=2e-5, batch_size=32, epochs=3
4. Creá un model card con descripción, métricas y licencia CC-BY-4.0
...
[continúa por cientos de líneas]
```

El problema es que ese prompt:

- ❌ No es reutilizable: hay que pegarlo en cada conversación nueva
- ❌ No es compartible: cada persona del equipo tiene su copia desactualizada
- ❌ No es mantenible: si cambia algo, hay que actualizar en decenas de lugares
- ❌ No tiene versiones: imposible saber qué cambió entre iteraciones

### La solución: la Skill

Una **Skill** empaqueta ese mismo conocimiento en un formato estructurado con un archivo `SKILL.md`:

```
mi-skill/
└── SKILL.md    ← metadatos + instrucciones
```

El archivo comienza con metadatos en YAML y luego contiene las instrucciones:

```yaml
---
name: "gradio-mediapipe"
description: "Construir interfaces web para modelos de visión con Gradio y MediaPipe."
---
# Instrucciones detalladas para el agente...
```

**Beneficios concretos:**

- ✓ Un solo lugar para mantener el conocimiento
- ✓ Versionable con git
- ✓ Compartible con todo el equipo
- ✓ Los agentes la cargan automáticamente cuando la necesitan

> El archivo `SKILL.md` que define el estilo de estos materiales *es en sí mismo una Skill*. Ya la estuvimos usando sin darnos cuenta.

*Fuente: [🤗 Context Course — Unit 1: What Are Skills?](https://huggingface.co/learn/context-course/unit1/what-are-skills)*

## Paso 1 — Instalación de herramientas

Vamos a instalar las tres bibliotecas que necesitamos para este notebook.

In [1]:
# Instala y verifica el entorno mínimo para visión artificial con interfaz web.
# Se ejecuta una sola vez por entorno; las celdas siguientes asumen que estas librerías están disponibles.

!pip install gradio mediapipe opencv-python-headless --quiet

# opencv-python-headless: variante sin dependencias de GUI (Qt/GTK); evita conflictos
# en entornos de servidor o Jupyter donde no hay pantalla física disponible.
# --quiet: suprime el log de descarga para no saturar la salida del notebook.

import gradio as gr   # Framework para construir la interfaz web interactiva (slider, imagen, botón).
import mediapipe as mp # Motor de detección de landmarks (manos, rostro, pose).
import cv2             # Procesamiento de imágenes: redimensionar, convertir color, dibujar sobre frames.
import numpy as np     # Las imágenes son arrays NumPy; mp y cv2 operan sobre ellos directamente.

# Se capturan las versiones antes de imprimir para poder reutilizarlas
# en condiciones o mensajes de error posteriores sin llamar a __version__ de nuevo.
version_gradio    = gr.__version__
version_mediapipe = mp.__version__
version_opencv    = cv2.__version__
version_numpy     = np.__version__

# Confirmación visual de que todas las importaciones fueron exitosas y el entorno es reproducible.
# Si algún import falla, esta celda lanza ImportError antes de llegar al print.
print("✓ Entorno listo.")
print(f"  gradio     {version_gradio}")
print(f"  mediapipe  {version_mediapipe}")
print(f"  opencv     {version_opencv}")
print(f"  numpy      {version_numpy}")



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Proyectos\rodriguez-carmen-pdi-1c-2026\.venv_vision_aplicada\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Entorno listo.
  gradio     6.19.0
  mediapipe  0.10.35
  opencv     4.13.0
  numpy      2.4.6


## Paso 2 — Patrón `gr.Interface`: envolver una función en segundos

`gr.Interface` es el camino más corto de una función Python a una interfaz web. Solo necesita tres argumentos:

1. `fn` — la función que procesa la entrada
2. `inputs` — el tipo de componente de entrada (imagen, texto, número...)
3. `outputs` — el tipo de componente de salida

### ✦ Paso manual: el flujo a mano

Antes de ejecutar, pensemos cómo funciona el recorrido completo:

```
Usuario sube imagen desde el navegador
         ↓
  gr.Interface la convierte a un array NumPy (H × W × 3, RGB)
         ↓
  Llama a nuestra función con ese array
         ↓
  La función devuelve un nuevo array NumPy (imagen procesada)
         ↓
  gr.Interface muestra el resultado en el navegador
```

Vamos a probar con un filtro de escala de grises: la función recibe la imagen en color y devuelve la misma imagen en grises, convertida de vuelta a 3 canales para que Gradio la muestre correctamente.

In [2]:
# Interfaz mínima con Gradio: el usuario sube una foto y recibe la misma imagen en escala de grises.
# Gradio actúa como servidor web local; no se necesita código HTML ni JavaScript.
import gradio as gr
import numpy as np
import cv2

def aplicar_filtro_gris(imagen_entrada):
    """
    Recibe una imagen en formato NumPy (alto x ancho x 3, RGB).
    Devuelve la misma imagen convertida a escala de grises, en 3 canales.
    """
    # Gradio entrega las imágenes en RGB (no BGR), por eso se usa COLOR_RGB2GRAY
    # y no COLOR_BGR2GRAY; un error aquí produciría un resultado visualmente idéntico
    # pero conceptualmente incorrecto si se encadenara con otras operaciones OpenCV.
    imagen_gris_un_canal = cv2.cvtColor(imagen_entrada, cv2.COLOR_RGB2GRAY)

    # cvtColor con COLOR_RGB2GRAY devuelve un array 2D (alto x ancho).
    # Gradio necesita un array 3D (alto x ancho x 3) para mostrar la imagen de salida;
    # se replica el canal gris tres veces para cumplir ese contrato sin alterar los valores.
    canal_rojo = imagen_gris_un_canal
    canal_verde = imagen_gris_un_canal
    canal_azul = imagen_gris_un_canal
    imagen_gris_tres_canales = np.stack([canal_rojo, canal_verde, canal_azul], axis=-1)
    # axis=-1: apila a lo largo del último eje, construyendo la dimensión de canales → (H, W, 3)

    return imagen_gris_tres_canales


interfaz_filtro = gr.Interface(
    fn=aplicar_filtro_gris,              # Función que Gradio llama cada vez que el usuario sube una imagen.
    inputs=gr.Image(label="Imagen original"),   # gr.Image entrega un array NumPy RGB a la función por defecto.
    outputs=gr.Image(label="Imagen en grises"), # Espera un array NumPy o PIL Image como retorno.
    title="Filtro de escala de grises",
    description="Subí una fotografía. La aplicación la va a convertir a escala de grises.",
    flagging_mode="never"  # Desactiva el botón "Flag" que Gradio muestra por defecto para recolectar ejemplos problemáticos.
)

# launch() inicia un servidor HTTP local y muestra el enlace en la salida de la celda.
# Si se ejecuta en Colab o un servidor remoto, Gradio genera además un túnel público temporal.
interfaz_filtro.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Proyectos\rodriguez-carmen-pdi-1c-2026\.venv_vision_aplicada\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Paso 3 — Patrón `gr.Blocks`: control total del layout

`gr.Blocks` nos da control completo sobre el diseño. Podemos definir exactamente dónde aparece cada componente, conectar múltiples funciones a múltiples botones y crear interfaces con filas, columnas y pestañas.

| | `gr.Interface` | `gr.Blocks` |
|---|---|---|
| Velocidad de desarrollo | ✓ Muy rápido | Más verboso |
| Control del layout | Fijo | ✓ Total |
| Múltiples funciones | Una sola | ✓ Varias |
| Eventos personalizados | Limitados | ✓ Completos |

Vamos a construir una calculadora simple para ver cómo se conectan los componentes con los eventos.

In [3]:
# Calculadora interactiva con gr.Blocks: demuestra el modelo de layout explícito de Gradio,
# donde los componentes se conectan con eventos en lugar de declararse como inputs/outputs fijos.
import gradio as gr

def calcular_resultado(numero_a, numero_b, operacion):
    """
    Recibe dos números y el nombre de la operación.
    Devuelve el resultado como texto.
    """
    if operacion == "Suma":
        resultado = numero_a + numero_b
    elif operacion == "Resta":
        resultado = numero_a - numero_b
    elif operacion == "Multiplicación":
        resultado = numero_a * numero_b
    elif operacion == "División":
        if numero_b == 0:
            # Retorno anticipado con mensaje amigable; evita que ZeroDivisionError
            # interrumpa la interfaz con un traceback en lugar de un texto legible.
            return "Error: no se puede dividir por cero"
        resultado = numero_a / numero_b
    else:
        # Rama defensiva: con gr.Radio no debería alcanzarse, pero protege
        # si la función se llama programáticamente con un valor inesperado.
        return "Operación no reconocida"

    # `operacion` ya es str (viene de Radio), pero la conversión explícita
    # deja claro que el f-string espera texto, no un objeto arbitrario.
    texto_operacion = str(operacion)
    texto_resultado = f"{numero_a} — {texto_operacion} — {numero_b} = {resultado}"
    return texto_resultado


# gr.Blocks es la API de layout explícito de Gradio (alternativa a gr.Interface).
# El bloque `with` actúa como árbol de componentes: todo lo que se instancie dentro
# queda registrado en la interfaz. El `as interfaz_calculadora` captura la referencia
# necesaria para llamar a `.launch()` fuera del bloque.
with gr.Blocks(title="Calculadora") as interfaz_calculadora:

    gr.Markdown("## Calculadora interactiva")
    gr.Markdown("Ingresá dos números, elegí la operación y presioná Calcular.")

    with gr.Row():
        # value= define el valor por defecto para que la interfaz sea usable
        # de inmediato sin que el usuario tenga que escribir antes de hacer clic.
        entrada_numero_a = gr.Number(label="Número A", value=10)
        entrada_numero_b = gr.Number(label="Número B", value=5)

    # gr.Radio restringe la entrada a opciones predefinidas, eliminando la necesidad
    # de validar el texto libre en calcular_resultado.
    # value="Suma" pre-selecciona una opción para que siempre haya una operación activa.
    selector_operacion = gr.Radio(
        choices=["Suma", "Resta", "Multiplicación", "División"],
        label="Operación",
        value="Suma"
    )

    boton_calcular = gr.Button("Calcular")
    salida_resultado = gr.Textbox(label="Resultado")

    # .click() conecta el evento del botón con la función Python.
    # `inputs` es una lista de referencias a componentes (no valores); Gradio lee
    # sus valores actuales en el momento del clic y los pasa en el mismo orden
    # que los parámetros de calcular_resultado.
    boton_calcular.click(
        fn=calcular_resultado,
        inputs=[entrada_numero_a, entrada_numero_b, selector_operacion],
        outputs=salida_resultado
    )

interfaz_calculadora.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


c:\Proyectos\rodriguez-carmen-pdi-1c-2026\.venv_vision_aplicada\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Paso 4 — Integración: MediaPipe dentro de Gradio

Ahora combinamos lo que ya conocemos. La función que detecta landmarks con MediaPipe se convierte en la función central de la interfaz Gradio.

El flujo es el mismo que en el Paso 2, pero ahora la función hace algo más interesante:

```
Usuario sube fotografía
         ↓
  Gradio la convierte a NumPy RGB
         ↓
  Nuestra función la procesa con FaceLandmarker de MediaPipe
  (478 puntos clave faciales)
         ↓
  Devolvemos la imagen con los landmarks dibujados
         ↓
  Gradio muestra la imagen anotada en el navegador
```

Usamos **FaceLandmarker** de MediaPipe (Tasks API) — el sucesor de Face Mesh, compatible con MediaPipe 0.10+. La celda descarga el modelo `.task` si no está disponible localmente y luego inicia la interfaz.

In [4]:
# Detecta hasta 2 rostros con FaceLandmarker (Tasks API, MediaPipe 0.10+)
# y devuelve la imagen con los 478 puntos clave dibujados usando cv2.
import os, urllib.request
import gradio as gr
import mediapipe as mp
import cv2
import numpy as np


# ── Descarga del modelo ──────────────────────────────────────────────────

MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
MODEL_PATH = "face_landmarker.task"

if not os.path.exists(MODEL_PATH):
    print("Descargando modelo face_landmarker.task …")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

tamaño_mb = os.path.getsize(MODEL_PATH) / 1_048_576
print(f"✓ Modelo disponible: {tamaño_mb:.1f} MB")


# ── Configuración del detector ───────────────────────────────────────────

# Aliases para acortar el código; mismo patrón que en el notebook de manos.
FaceLandmarker        = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
BaseOptions           = mp.tasks.BaseOptions
RunningMode           = mp.tasks.vision.RunningMode

# RunningMode.IMAGE procesa cada llamada de forma independiente (sin tracker entre cuadros);
# es el modo correcto para imágenes sueltas, a diferencia de VIDEO o LIVE_STREAM.
opciones_detector = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=RunningMode.IMAGE,
    num_faces=2,                        # detecta hasta 2 rostros por imagen
    min_face_detection_confidence=0.5,  # descarta detecciones con probabilidad < 50 %
    min_face_presence_confidence=0.5    # umbral de presencia del rostro en el encuadre
)

# El detector se crea una sola vez fuera de la función para no re-inicializar el modelo
# en cada imagen que procese la interfaz (sería ineficiente a partir de la segunda llamada).
detector_facial = FaceLandmarker.create_from_options(opciones_detector)
print("✓ Detector de Face Mesh inicializado.")


# ── Función principal ────────────────────────────────────────────────────

def detectar_landmarks_faciales(imagen_entrada):
    """
    Recibe una imagen RGB como array NumPy (entregada por Gradio).
    Detecta los 478 puntos clave faciales con FaceLandmarker (Tasks API).
    Devuelve la imagen con los puntos dibujados.
    """
    alto, ancho = imagen_entrada.shape[:2]

    # La Tasks API necesita mp.Image; Gradio entrega RGB que coincide con SRGB.
    imagen_mp = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_entrada)
    resultado = detector_facial.detect(imagen_mp)

    # Se trabaja sobre una copia para no modificar el array original in-place.
    imagen_anotada = imagen_entrada.copy()

    # resultado.face_landmarks es una lista vacía [] cuando no hay rostros
    # (a diferencia de la API antigua que devolvía None).
    if not resultado.face_landmarks:
        print("No se detectó ningún rostro en la imagen.")
        return imagen_anotada

    print(f"Rostros detectados: {len(resultado.face_landmarks)}")

    for puntos_rostro in resultado.face_landmarks:
        # drawing_utils no está disponible en MediaPipe 0.10+; se dibuja con cv2 directamente.
        # Las coordenadas de cada landmark son normalizadas [0.0, 1.0] y se convierten a píxeles.
        for lm in puntos_rostro:
            px = int(lm.x * ancho)
            py = int(lm.y * alto)
            cv2.circle(imagen_anotada, (px, py), 1, (0, 220, 180), -1)
            # Radio 1 y relleno (-1): los 478 puntos se dibujan compactos para no superponerse.

    return imagen_anotada


# ── Interfaz Gradio ──────────────────────────────────────────────────────

interfaz_landmarks = gr.Interface(
    fn=detectar_landmarks_faciales,
    inputs=gr.Image(
        label="Fotografía",
        type="numpy"
        # type="numpy" es obligatorio: sin él Gradio entregaría PIL Image en lugar del
        # array NumPy que mp.Image() y cv2 requieren.
    ),
    outputs=gr.Image(label="Landmarks detectados"),
    title="Detector de Landmarks Faciales — Face Mesh",
    description=(
        "Subí una fotografía con uno o dos rostros. "
        "La aplicación detecta los 478 puntos clave del Face Mesh de MediaPipe "
        "y los dibuja sobre la imagen."
    ),
    flagging_mode="never"   # Desactiva el botón Flag para recolección de ejemplos.
)

# share=False: la interfaz corre solo en localhost; no se genera túnel público.
# Para compartir con otros equipos de la red se puede cambiar a share=True.
interfaz_landmarks.launch(share=False)

print()
print("✓ Interfaz activa.")
print("  Abrí el link de arriba en el navegador para usar la aplicación.")


✓ Modelo disponible: 3.6 MB
✓ Detector de Face Mesh inicializado.
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.



✓ Interfaz activa.
  Abrí el link de arriba en el navegador para usar la aplicación.


c:\Proyectos\rodriguez-carmen-pdi-1c-2026\.venv_vision_aplicada\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Rostros detectados: 2


## Para explorar

Estas son las fuentes que usamos como base para este notebook:

- [🤗 Context Course — Unit 1: What Are Skills?](https://huggingface.co/learn/context-course/unit1/what-are-skills) — El concepto de Skill explicado con ejemplos de agentes
- [🤗 huggingface-gradio Skill (GitHub)](https://github.com/huggingface/skills/tree/main/skills/huggingface-gradio) — El Skill oficial de Hugging Face para construir interfaces Gradio
- [Documentación oficial de Gradio](https://www.gradio.app/docs) — Referencia completa de componentes, layouts y eventos
- [MediaPipe Face Landmarker](https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker) — Documentación del modelo de landmarks faciales

---

### ✎ Para pensar

1. **Sobre `gr.Interface` vs `gr.Blocks`:** Revisá el detector de landmarks que construimos. ¿Qué tendríamos que cambiar para agregar un segundo botón que, en lugar de detectar landmarks, aplique el filtro de grises del Paso 2? ¿Alcanzaría con `gr.Interface` o necesitaríamos `gr.Blocks`? ¿Por qué?

2. **Sobre el concepto de Skill:** Revisá el archivo `SKILL.md` que guía el estilo de estos materiales. ¿Qué ventajas concretas tiene tener ese conocimiento en un archivo separado en lugar de escribirlo en el prompt de cada conversación?

3. **Sobre la integración:** La función `detectar_landmarks_faciales` recibe y devuelve arrays NumPy. ¿Qué habría que cambiar para que la entrada fuera un video en lugar de una imagen? ¿Qué componente de Gradio usarías? ¿La función de procesamiento cambiaría en algo?